##Step 02 - FAISS In-Memory Index

In [ ]:
print("Embedding dimension:", len(all_vectors[0]))
print("EMBEDDING_DIM:", EMBEDDING_DIM)

Embedding dimension: 384
EMBEDDING_DIM: 384


In [ ]:
import faiss as faiss_lib

print("Building FAISS IndexFlatL2...")
t0 = time.perf_counter()
raw_index = faiss_lib.IndexFlatL2(EMBEDDING_DIM)
vecs_np   = np.array(all_vectors, dtype=np.float32)
raw_index.add(vecs_np)
build_ms  = (time.perf_counter() - t0) * 1000
print(f"FAISS IndexFlatL2 built in {build_ms:.1f}ms | {raw_index.ntotal} vectors")

Building FAISS IndexFlatL2...
FAISS IndexFlatL2 built in 2.4ms | 40 vectors


In [ ]:
# LangChain wrapper for convenient .invoke() retrieval
faiss_store     = FAISS.from_documents(all_docs, embedder)
faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": TOP_K})
print("LangChain FAISS vectorstore ready")

# Demo search
sample_q = "How does vector similarity search work?"
results  = faiss_retriever.invoke(sample_q)
print(f"\nQuery: '{sample_q}'")
for i, r in enumerate(results):
    print(f"  [{i+1}] [{r.metadata['title']}] {r.page_content[:100]}...")


LangChain FAISS vectorstore ready

Query: 'How does vector similarity search work?'
  [1] [Vector Database] A vector database stores data as high-dimensional vectors representing text, images, audio, or video...
  [2] [FAISS] FAISS (Facebook AI Similarity Search) is an open-source library by Meta AI for efficient similarity ...
  [3] [BM25 Information Retrieval] . It excels at exact keyword matching and handling rare terms. Weakness: vocabulary mismatch, no sem...
  [4] [Retrieval-Augmented Generation] . (2) Embed the user query and retrieve top-k similar chunks. (3) Inject chunks into the prompt. (4)...
  [5] [Azure AI Search] Azure AI Search (formerly Azure Cognitive Search) is a fully managed cloud search service from Micro...


In [ ]:
BENCHMARK_QUERIES = [
    "What is retrieval-augmented generation?",
    "How does HNSW approximate nearest neighbour search work?",
    "What are the differences between BM25 and vector search?",
    "How does CRISPR-Cas9 cut DNA?",
    "What caused the COVID-19 pandemic?",
    "How do electric vehicles use regenerative braking?",
    "What is the transformer attention mechanism?",
    "What are the risks of climate change?",
    "How does Proof of Work consensus work in Bitcoin?",
    "What is FAISS and when should I use it?",
    "How does Pinecone handle metadata filtering?",
    "What is hybrid search combining BM25 and vectors?",
    "How does Azure AI Search differ from Pinecone?",
    "What are embedding models used for in RAG?",
    "How does quantum computing threaten RSA encryption?",
    "What is the supply chain bullwhip effect?",
    "How does renewable energy handle intermittency?",
    "What cybersecurity threats target cloud infrastructure?",
    "What is the NIST cybersecurity framework?",
    "How do word embeddings capture semantic meaning?",
]
print(f"{len(BENCHMARK_QUERIES)} benchmark queries ready")


20 benchmark queries ready


In [ ]:
faiss_latencies = []
for q in tqdm(BENCHMARK_QUERIES, desc="FAISS benchmark"):
    t0 = time.perf_counter()
    faiss_retriever.invoke(q)
    faiss_latencies.append((time.perf_counter() - t0) * 1000)

faiss_p50 = float(np.percentile(faiss_latencies, 50))
faiss_p95 = float(np.percentile(faiss_latencies, 95))
print(f"FAISS  p50={faiss_p50:.1f}ms  p95={faiss_p95:.1f}ms (includes embedding query)")

FAISS benchmark:   0%|          | 0/20 [00:00<?, ?it/s]

FAISS  p50=26.9ms  p95=93.9ms (includes embedding query)
